# **04 - Conformal Prediction Application**

## **Summary**

---
## **Table of Contents**

---
## **Objective**

## <a id ='data-info'></a> **Dataset Info**
* **Source:** UCI Machine Learning Repository
* **Title:** Appliances Energy Prediction
* **Time Range:** 11th January 2016 - 27th May 2016
* **Primary Target Variable:** `Appliances` (Watt-hour consumed)

## **Initial Setup**

In [ ]:
import pandas as pd
from pathlib import Path
import sys
from sklearn.model_selection import train_test_split
import joblib
import numpy as np
from aci.tracker import ACITracker
import matplotlib.pyplot as plt
import numpy as np
from collections import deque
import math

# Add the root directory (one level up from notebooks/) to Python's path
root_dir = Path.cwd().parent
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))

from src.preprocess import preprocess_energy, feature_engineering, xgb_preprocess

In [ ]:
PROJECT_ROOT = Path.cwd().parent

# Import csv file
energy = pd.read_csv(f'{PROJECT_ROOT}/data/raw/energydata_complete.csv')

In [ ]:
# Data preprocessing
energy = preprocess_energy(energy)

# Form feature through feature engineering
energy = feature_engineering(energy)

ENCODER_PATH = root_dir / "models" / "month_encoder.pkl"

encoder = joblib.load(ENCODER_PATH)

# Separate feature from target and preprocess predictors
X = xgb_preprocess(energy, encoder)
y = energy['Appliances']


# Train-calibration-test split (70-15-15)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, shuffle= False
)

X_cal, X_test, y_cal, y_test = train_test_split(
    X_test, y_test, shuffle= False
)

## **Import Model**

In [ ]:
MODEL_PATH = PROJECT_ROOT/"models"/"xgb_best.pkl"

# Use joblib.load instead of pickle.load
model = joblib.load(MODEL_PATH)

## **Make Predictions**

In [ ]:
# Predict calibration set
y_pred_cal = np.asarray(model.predict(X_cal))
y_pred_cal = pd.Series(
    y_pred_cal, 
    index=y_cal.index
)

In [ ]:
# Predict test set
y_pred_test = np.asarray(model.predict(X_test))
y_pred_test = pd.Series(
    y_pred_test, 
    index=y_test.index
)

## **Apply Adaptive Conformal Prediction**

In [ ]:
# Calculate calibration residuals
cal_residuals = np.abs(y_cal - y_pred_cal)

In [ ]:
# Tracker properties
TARGET_COVERAGE = 0.95
ALPHA = 1 - TARGET_COVERAGE
GAMMA = 0.05

# Initialize tracker
tracker = ACITracker(alpha=ALPHA, gamma=GAMMA, method= 'simple')

In [ ]:
def get_conformal_quantile(residuals, alpha_t):
    """
    Calculate conformal prediction interval based on past residual.
    """

    # Convert residual history into NumPy array
    residuals = np.asarray(residuals, dtype=float)

    # Get the number of residuals present
    n = len(residuals)

    # Keep alpha in a range that gives a finite order statistic.
    alpha_min = 1.0 / (n + 1)
    alpha_max = n / (n + 1)

    alpha_effective = float(
        np.clip(alpha_t, alpha_min, alpha_max)
    )

    k = math.ceil((n + 1) * (1.0 - alpha_effective))

    # Calculate interval quantile (q)
    q = np.partition(residuals, k - 1)[k - 1]

    return q, alpha_t

In [ ]:
actual = y_test.to_numpy(dtype=float)
predicted = y_pred_test.to_numpy(dtype=float)

n_test = len(actual)

lower = np.full(n_test, np.nan)
upper = np.full(n_test, np.nan)
width = np.full(n_test, np.nan)
covered = np.full(n_test, np.nan)

# Raw alpha is the value held by the package tracker.
alpha_raw = np.full(n_test, np.nan)

# Effective alpha is the locally clipped value used for quantiles.
alpha_effective = np.full(n_test, np.nan)

quantiles = np.full(n_test, np.nan)

for t in range(n_test):

    # A. Capture the current tracker state before prediction.
    alpha_raw[t] = tracker.alpha_t

    # Skip invalid predictions. They cannot produce a valid interval.
    if not np.isfinite(predicted[t]):
        continue

    # B. Compute the interval using past residuals only.
    q_t, alpha_effective[t] = get_conformal_quantile(
        list(cal_residuals),
        alpha_raw[t]
    )

    quantiles[t] = q_t

    # C. Construct the prediction interval.
    lower[t] = max(0, predicted[t] - q_t)
    upper[t] = predicted[t] + q_t
    width[t] = upper[t] - lower[t]

    # D. Reveal the actual value and evaluate coverage.
    if not np.isfinite(actual[t]):
        continue

    covered[t] = float(
        lower[t] <= actual[t] <= upper[t]
    )

    err_t = 1.0 - covered[t]

    # E. Update the package tracker for the NEXT observation.
    tracker.update(err_t)

    # F. Update the rolling residual buffer AFTER evaluation.
    # This residual cannot influence the interval at time t.
    new_residual = abs(actual[t] - predicted[t])
    
    np.append(cal_residuals, new_residual)


# ============================================================
# 6. CREATE RESULTS TABLE AND EVALUATION METRICS
# ============================================================

results = pd.DataFrame(
    {
        "actual": actual,
        "predicted": predicted,
        "lower": lower,
        "upper": upper,
        "width": width,
        "covered": covered,
        "alpha_raw": alpha_raw,
        "alpha_effective": alpha_effective,
        "quantile": quantiles,
    },
    index=y_test.index
)

# Only evaluate observations with finite actual values,
# predictions, and interval boundaries.
valid = (
    np.isfinite(results["actual"])
    & np.isfinite(results["predicted"])
    & np.isfinite(results["lower"])
    & np.isfinite(results["upper"])
    & np.isfinite(results["covered"])
)

evaluated = results.loc[valid]


empirical_coverage = evaluated["covered"].mean()
mean_width = evaluated["width"].mean()
median_width = evaluated["width"].median()
miscoverage_rate = 1.0 - empirical_coverage

print("=" * 55)
print("ADAPTIVE CONFORMAL INFERENCE RESULTS")
print("=" * 55)
print(f"Target coverage:       {TARGET_COVERAGE:.2%}")
print(f"Empirical coverage:    {empirical_coverage:.2%}")
print(f"Miscoverage rate:      {miscoverage_rate:.2%}")
print(f"Mean interval width:   {mean_width:.4f}")
print(f"Median interval width: {median_width:.4f}")
print(f"Valid test points:     {len(evaluated)}")
print(f"Target alpha:          {ALPHA:.2f}")
print(f"Gamma:                 {GAMMA:.2f}")

In [ ]:
# ============================================================
# 7. PLOT 1 — ACTUAL, PREDICTED AND CONFORMAL INTERVAL
# ============================================================

fig, axes = plt.subplots(
    3, 1,
    figsize=(16, 16),
    sharex=True,
    gridspec_kw={
        "height_ratios": [3, 1.5, 1.5]
    }
)

# Plot 1: Prediction intervals
ax = axes[0]

ax.fill_between(
    results.index,
    results["lower"].to_numpy(),
    results["upper"].to_numpy(),
    alpha=0.25,
    label="ACI prediction interval"
)

ax.plot(
    results.index,
    results["actual"],
    label="Actual value",
    linewidth=1.2
)

ax.plot(
    results.index,
    results["predicted"],
    label="Predicted value",
    linewidth=1.2
)

# Highlight observations outside the prediction interval.
missed = (
    valid
    & (results["covered"] == 0)
)

ax.scatter(
    results.index[missed],
    results.loc[missed, "actual"],
    marker="x",
    s=45,
    label="Missed observation",
    zorder=4
)

ax.set_title(
    "ACI: Actual vs Predicted with Adaptive Conformal Intervals"
)
ax.set_ylabel("Target value")
ax.legend(loc="best")
ax.grid(True, alpha=0.3)


# ============================================================
# 8. PLOT 2 — ROLLING EMPIRICAL COVERAGE
# ============================================================

ax = axes[1]

ax.plot(
    results.index,
    results["alpha_raw"],
    label="Raw tracker alpha_t",
    linewidth=1.0,
    alpha=0.8
)

ax.plot(
    results.index,
    results["alpha_effective"],
    label="Effective alpha used for quantile",
    linewidth=1.2
)

ax.axhline(
    0.05,
    linestyle="--",
    label="Target alpha (0.05)"
)

ax.set_ylabel("Miscoverage level")
ax.set_title("ACI Adaptive Miscoverage Level")
ax.legend(loc="best")
ax.grid(True, alpha=0.3)


# ============================================================
# 10. PLOT 4 — INTERVAL WIDTH
# ============================================================

ax = axes[2]

ax.plot(
    results.index,
    results["width"],
    label="Prediction interval width",
    linewidth=1.2
)

ax.axhline(
    mean_width,
    linestyle="--",
    label=f"Mean width ({mean_width:.3f})"
)

ax.set_xlabel("Time")
ax.set_ylabel("Upper - lower")
ax.set_title("Evolution of Prediction Interval Width")
ax.legend(loc="best")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()